# 05 - Hyperparameter tuning
**Milestone 5 (spec 3.5)** | Owner: Experiment and Tuning Engineer (role 5). Everything is chosen on the **validation** set; the test set stays untouched. Seed 42, single run per setting (the 3-seed check is in the final evaluation).
What we tune and why:
1. **VGG16 fine-tuning** (best model in M4): fine-tune learning rate {1e-5, 3e-5} and dropout {0.3, 0.5}. One setting is the M4 reference (1e-5, 0.3).
2. **Small CNN**: M4 showed an unstable validation loss, so we add **BatchNorm** after each conv (it normalizes activations and usually makes training steadier) and compare learning rates {1e-3, 3e-4}.
3. **Decision threshold**: recall on pneumonia is the primary metric, so we pick the cut-off probability that reaches a recall target while losing as little specificity as possible. The threshold is chosen on val and then frozen.
**Runtime: T4 GPU.**

In [ ]:
import os, sys, time
WORK = '/content/work' if os.path.exists('/content') else '..'
os.makedirs(WORK + '/src', exist_ok=True)
open(WORK + '/src/__init__.py', 'w').write('')
open(WORK + '/src/data.py', 'w').write('"""Data loading and inventory for the Chest X-Ray (Pneumonia) project.\n\nMilestone 1 only needs inventory + EDA helpers. Splitting and the tf.data\npipeline are added in Milestone 2 in this same module.\n"""\nimport hashlib\nimport os\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nDATASET = "paultimothymooney/chest-xray-pneumonia"\nCLASSES = ["NORMAL", "PNEUMONIA"]  # label 0 = NORMAL, label 1 = PNEUMONIA (positive class)\nIMG_EXT = {".jpeg", ".jpg", ".png"}\n\n\ndef download(dataset: str = DATASET) -> str:\n    """Download the dataset with kagglehub and return the local folder."""\n    import kagglehub\n    return kagglehub.dataset_download(dataset)\n\n\ndef find_data_root(base: str) -> Path:\n    """Find the folder that directly contains train/, test/ and val/.\n\n    The Kaggle zip nests the data (chest_xray/chest_xray/...) and also holds a\n    __MACOSX copy, so we search instead of hard-coding a path.\n    """\n    for p in sorted(Path(base).rglob("train")):\n        if p.is_dir() and "__MACOSX" not in p.parts and (p.parent / "test").is_dir():\n            return p.parent\n    raise FileNotFoundError(f"No train/test folders under {base}")\n\n\ndef file_md5(path) -> str:\n    """MD5 of the raw file bytes, used to detect exact duplicate images."""\n    h = hashlib.md5()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef build_inventory(root, with_hash: bool = True) -> pd.DataFrame:\n    """One row per image: path, original split, label, size, mode, file size, hash.\n\n    The label comes from the parent folder name. Pneumonia files are named\n    person<ID>_bacteria_<n> / person<ID>_virus_<n>, which gives a subtype and a\n    patient id (Normal files are IM-xxxx-yyyy / NORMAL2-IM-xxxx-yyyy and carry no\n    patient id we can rely on).\n    """\n    rows = []\n    for split in ["train", "val", "test"]:\n        for cls in CLASSES:\n            d = Path(root) / split / cls\n            if not d.is_dir():\n                continue\n            for p in sorted(d.iterdir()):\n                if p.suffix.lower() not in IMG_EXT or p.name.startswith("."):\n                    continue\n                with Image.open(p) as im:\n                    w, h = im.size\n                    mode = im.mode\n                name = p.name.lower()\n                subtype = "bacteria" if "bacteria" in name else "virus" if "virus" in name else "normal"\n                patient = name.split("_")[0] if name.startswith("person") else None\n                rows.append(dict(\n                    path=str(p), orig_split=split, label_name=cls, label=CLASSES.index(cls),\n                    subtype=subtype, patient_id=patient, width=w, height=h, mode=mode,\n                    kb=os.path.getsize(p) / 1024,\n                    md5=file_md5(p) if with_hash else None))\n    df = pd.DataFrame(rows)\n    df["aspect"] = df["width"] / df["height"]\n    df["megapixels"] = df["width"] * df["height"] / 1e6\n    return df\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--download", action="store_true")\n    ap.add_argument("--root", default=None)\n    a = ap.parse_args()\n    base = download() if a.download else a.root\n    root = find_data_root(base)\n    inv = build_inventory(root)\n    print(root, len(inv))\n    print(inv.groupby(["orig_split", "label_name"]).size())\n\n\n# ---------------------------------------------------------------------------\n# Milestone 2: cleaning, leak-free splitting, class weights, tf.data pipeline\n# ---------------------------------------------------------------------------\nimport numpy as np\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nIMG_SIZE = 224\nSEED = 42\n\n\ndef drop_exact_duplicates(df: pd.DataFrame) -> pd.DataFrame:\n    """Keep one copy of every byte-identical image (same MD5).\n\n    If the same bytes carry different labels the label is ambiguous, so all copies are dropped.\n    """\n    lab_per_hash = df.groupby("md5")["label"].nunique()\n    conflicting = lab_per_hash[lab_per_hash > 1].index\n    out = df[~df["md5"].isin(conflicting)]\n    return out.drop_duplicates("md5", keep="first").reset_index(drop=True)\n\n\ndef add_groups(df: pd.DataFrame) -> pd.DataFrame:\n    """Add `group`: images of the same patient must share a group.\n\n    Pneumonia files carry a patient id in the name. NORMAL files do not, so each\n    NORMAL image is its own group (limitation: a NORMAL patient may still span splits).\n    """\n    df = df.copy()\n    df["group"] = np.where(df["patient_id"].notna(), "P_" + df["patient_id"].astype(str), "I_" + df["md5"])\n    return df\n\n\ndef make_splits(df: pd.DataFrame, seed: int = SEED, n_folds: int = 20,\n                n_test: int = 3, n_val: int = 3) -> pd.DataFrame:\n    """Stratified, group-aware 70/15/15 split (14 train folds, 3 val, 3 test of 20).\n\n    StratifiedGroupKFold keeps the class ratio in every fold AND never puts one group\n    (patient) in two folds. Cutting 20 folds then merging them gives ~70/15/15.\n    """\n    df = add_groups(df)\n    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)\n    fold = np.full(len(df), -1)\n    for k, (_, idx) in enumerate(sgkf.split(df, df["label"], df["group"])):\n        fold[idx] = k\n    df["split"] = np.where(fold < n_test, "test", np.where(fold < n_test + n_val, "val", "train"))\n    return df\n\n\ndef check_no_leakage(df: pd.DataFrame) -> None:\n    """Raise AssertionError if any group, file or hash appears in two splits."""\n    for col in ["group", "md5", "path"]:\n        n = df.groupby(col)["split"].nunique()\n        assert (n <= 1).all(), f"Leakage: {int((n > 1).sum())} {col} values span several splits"\n    assert set(df["split"]) == {"train", "val", "test"}\n\n\ndef class_weights(train_df: pd.DataFrame) -> dict:\n    """Balanced weights w_c = N / (n_classes * n_c), computed from TRAIN only."""\n    counts = train_df["label"].value_counts().sort_index()\n    n, k = len(train_df), len(counts)\n    return {int(c): float(n / (k * m)) for c, m in counts.items()}\n\n\ndef make_dataset(df: pd.DataFrame, batch_size: int = 32, img_size: int = IMG_SIZE,\n                 training: bool = False, augment: bool = False, seed: int = SEED):\n    """tf.data pipeline: file -> 3-channel float32 [0,255] resized image, label.\n\n    Pixels are left in [0,255]; the *model* rescales them (Rescaling(1/255) for the custom CNN,\n    preprocess_input for VGG16/ResNet-50). That keeps training and inference preprocessing identical.\n    Augmentation (flip is deliberately excluded for X-rays) runs on the TRAIN set only.\n    """\n    import tensorflow as tf\n\n    def load(path, y):\n        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)\n        img = tf.image.resize(img, (img_size, img_size))\n        return tf.cast(img, tf.float32), tf.cast(y, tf.float32)\n\n    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values))\n    if training:\n        ds = ds.shuffle(len(df), seed=seed, reshuffle_each_iteration=True)\n    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)\n    if augment:\n        aug = tf.keras.Sequential([\n            tf.keras.layers.RandomRotation(0.04, seed=seed),\n            tf.keras.layers.RandomZoom(0.1, seed=seed),\n            tf.keras.layers.RandomContrast(0.15, seed=seed),\n        ])\n        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)\n    return ds.prefetch(tf.data.AUTOTUNE)\n\n\ndef load_flat_arrays(df: pd.DataFrame, size: int = 64) -> np.ndarray:\n    """Load images as flat grey vectors (size*size) in [0,1] for the classical baselines."""\n    out = np.empty((len(df), size * size), dtype=np.float32)\n    for i, p in enumerate(df["path"].values):\n        with Image.open(p) as im:\n            out[i] = np.asarray(im.convert("L").resize((size, size)), dtype=np.float32).ravel() / 255.0\n    return out\n\n\ndef rebase_paths(df: pd.DataFrame, root) -> pd.DataFrame:\n    """Point `path` at the current dataset folder.\n\n    Colab starts a fresh machine every session, so absolute paths saved in a CSV from an\n    earlier session are stale. The layout <split>/<class>/<file> is stable, so rebuild from it.\n    """\n    df = df.copy()\n    split_col = "orig_split"\n    df["path"] = [str(Path(root) / s / c / Path(p).name)\n                  for s, c, p in zip(df[split_col], df["label_name"], df["path"])]\n    return df\n')
open(WORK + '/src/evaluate.py', 'w').write('"""Metrics shared by baselines, neural models and the final test evaluation."""\nimport numpy as np\nfrom sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,\n                             precision_score, recall_score, roc_auc_score)\n\n\ndef binary_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:\n    """Accuracy, per-class precision/recall/F1, macro-F1 and ROC-AUC for the binary task.\n\n    Positive class = PNEUMONIA (1). `recall_pneumonia` is the PRIMARY metric;\n    `recall_normal` is the specificity (how many healthy children are correctly cleared).\n    """\n    y_true = np.asarray(y_true).astype(int)\n    y_prob = np.asarray(y_prob, dtype=float)\n    y_pred = (y_prob >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) == 2 and len(set(y_prob)) > 1 else float("nan")\n    return dict(\n        accuracy=accuracy_score(y_true, y_pred),\n        recall_pneumonia=recall_score(y_true, y_pred, pos_label=1, zero_division=0),\n        recall_normal=recall_score(y_true, y_pred, pos_label=0, zero_division=0),\n        precision_pneumonia=precision_score(y_true, y_pred, pos_label=1, zero_division=0),\n        precision_normal=precision_score(y_true, y_pred, pos_label=0, zero_division=0),\n        f1_pneumonia=f1_score(y_true, y_pred, pos_label=1, zero_division=0),\n        f1_normal=f1_score(y_true, y_pred, pos_label=0, zero_division=0),\n        macro_f1=f1_score(y_true, y_pred, average="macro", zero_division=0),\n        auc=auc, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))\n')
open(WORK + '/src/train.py', 'w').write('"""Experiment logging (spec 3.5). The training loop is added in Milestone 4."""\nimport csv\nimport datetime as dt\nimport os\n\nLOG_COLUMNS = ["run_id", "date", "author", "model", "hyperparams", "seed", "train_time_s",\n               "params", "split", "accuracy", "recall_pneumonia", "recall_normal",\n               "precision_pneumonia", "f1_pneumonia", "macro_f1", "auc", "notes"]\n\n\ndef log_run(path: str, author: str, model: str, hyperparams: dict, seed: int, train_time_s: float,\n            params, split: str, metrics: dict, notes: str = "") -> str:\n    """Append one row to the experiment CSV (creates it with a header) and return the run id."""\n    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n    new = not os.path.exists(path)\n    n = 1 if new else sum(1 for _ in open(path)) \n    run_id = f"R{n:03d}"\n    row = dict(run_id=run_id, date=dt.datetime.now().strftime("%Y-%m-%d %H:%M"), author=author, model=model,\n               hyperparams=str(hyperparams), seed=seed, train_time_s=round(train_time_s, 1), params=params,\n               split=split, notes=notes, **{k: (round(v, 4) if isinstance(v, float) else v)\n                                           for k, v in metrics.items() if k in LOG_COLUMNS})\n    with open(path, "a", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)\n        if new:\n            w.writeheader()\n        w.writerow(row)\n    return run_id\n')
if os.path.exists('/content'):
    %cd /content/work
sys.path.insert(0, os.path.abspath('.'))
!pip -q install kagglehub
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from src.data import *
from src.evaluate import binary_metrics
from src.train import log_run
tf.keras.utils.set_random_seed(SEED)
print('GPU:', tf.config.list_physical_devices('GPU'))
LOG = 'experiments/experiment_log.csv'
AUTHOR = 'tuning-engineer'
os.makedirs('models', exist_ok=True)

In [ ]:
ROOT = find_data_root(download())
inv = build_inventory(ROOT)
df = make_splits(drop_exact_duplicates(inv)); check_no_leakage(df)   # same seed as notebook 02, same split
tr, va = df[df.split == 'train'], df[df.split == 'val']
cw = class_weights(tr); yva = va.label.values
# Decode the images once and keep them in RAM (.cache); augmentation is applied after the cache so it stays random.
raw_tr = make_dataset(tr, 32, training=True).unbatch().cache().shuffle(len(tr), seed=42, reshuffle_each_iteration=True).batch(32)   # cache single images, re-shuffle every epoch
aug = keras.Sequential([layers.RandomRotation(0.04, seed=SEED), layers.RandomZoom(0.1, seed=SEED), layers.RandomContrast(0.15, seed=SEED)])
train_ds = raw_tr.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = make_dataset(va, 32).cache()
print(len(tr), 'train |', len(va), 'val | class weights', cw)

## 1. VGG16: fine-tune learning rate x dropout
Each run has two phases like M4: (1) train only the new head for 3 epochs at lr 1e-3, (2) unfreeze `block5` and train at the tested learning rate (max 8 epochs, stop after 2 epochs without val-loss improvement and keep the best weights).

In [ ]:
def build_vgg(dropout):
    base = keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Lambda(keras.applications.vgg16.preprocess_input)(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(x)), base

RESULTS = {}
def run_vgg(lr_ft, dropout):
    name = f'vgg16_ft_lr{lr_ft:g}_do{dropout:g}'
    tf.keras.utils.set_random_seed(SEED)
    model, base = build_vgg(dropout); t0 = time.time()
    model.compile(keras.optimizers.Adam(1e-3), 'binary_crossentropy')
    model.fit(train_ds, validation_data=val_ds, epochs=3, class_weight=cw, verbose=0)
    for l in base.layers: l.trainable = l.name.startswith('block5')
    base.trainable = True
    model.compile(keras.optimizers.Adam(lr_ft), 'binary_crossentropy')
    h = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=cw, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_loss')])
    secs = time.time() - t0
    p = model.predict(val_ds, verbose=0).ravel(); m = binary_metrics(yva, p)
    trainable = int(sum(np.prod(v.shape) for v in model.trainable_variables))
    log_run(LOG, AUTHOR, name, dict(lr_ft=lr_ft, dropout=dropout, head_epochs=3, ft_epochs=len(h.history['loss'])), SEED, secs, model.count_params(), 'val', m, notes=f'trainable={trainable}')
    RESULTS[name] = p
    print(name, 'epochs', len(h.history['loss']), 'best val loss', round(min(h.history['val_loss']), 4), {k: round(float(v), 4) for k, v in m.items() if k in ('accuracy', 'recall_pneumonia', 'recall_normal', 'macro_f1', 'auc', 'fn')}, round(secs), 's')
    return model

for lr_ft, do in [(1e-5, 0.3), (3e-5, 0.3), (1e-5, 0.5), (3e-5, 0.5)]:
    mdl = run_vgg(lr_ft, do)
    keras.backend.clear_session()

## 2. Small CNN with BatchNorm
Same 3-block CNN as M4 (16, 32, 64 filters) but `Conv -> BatchNorm -> ReLU -> MaxPool`. BatchNorm adds 2 trainable parameters per channel (gamma, beta) plus 2 non-trainable running statistics: per conv layer that is 4 x filters. Extra = 4 x (16 + 32 + 64) = 448, so the total becomes 27,809 + 448 = **28,257** (trainable 28,257 - 224 = 28,033).

In [ ]:
def build_cnn_bn():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1/255)(inp)
    for f in (16, 32, 64):
        x = layers.Conv2D(f, 3, padding='same')(x)
        x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x); x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(x))
assert build_cnn_bn().count_params() == 28257

for lr in (1e-3, 3e-4):
    name = f'cnn_bn_lr{lr:g}'
    tf.keras.utils.set_random_seed(SEED)
    cnn = build_cnn_bn(); cnn.compile(keras.optimizers.Adam(lr), 'binary_crossentropy'); t0 = time.time()
    h = cnn.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=cw, verbose=0,
                callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_loss')])
    secs = time.time() - t0
    p = cnn.predict(val_ds, verbose=0).ravel(); m = binary_metrics(yva, p)
    log_run(LOG, AUTHOR, name, dict(lr=lr, bn=True, dropout=0.3, epochs=len(h.history['loss'])), SEED, secs, cnn.count_params(), 'val', m)
    RESULTS[name] = p
    print(name, 'epochs', len(h.history['loss']), 'best val loss', round(min(h.history['val_loss']), 4), {k: round(float(v), 4) for k, v in m.items() if k in ('accuracy', 'recall_pneumonia', 'recall_normal', 'macro_f1', 'auc', 'fn')}, round(secs), 's')
    keras.backend.clear_session()

## 3. Comparison and decision threshold
Primary metric: recall on pneumonia. The M4 and M3 numbers (same val set) are given for reference.

In [ ]:
log = pd.read_csv(LOG)
cols = ['model', 'params', 'accuracy', 'recall_pneumonia', 'recall_normal', 'f1_pneumonia', 'macro_f1', 'auc', 'train_time_s']
display(log[cols].round(3))
print('REFERENCE (M3/M4, val): MLP acc .953 recall_pneu .969 recall_norm .911 auc .987 | cnn_small acc .824 recall_pneu .809 | vgg16_finetune_block5 (lr 1e-5, do .3) acc .958 recall_pneu .962 recall_norm .945 auc .990')
best = log[log.model.str.startswith('vgg16_ft')].sort_values(['auc'], ascending=False).iloc[0].model
print('best VGG setting by val AUC:', best)

In [ ]:
p = RESULTS[best]; rows = []
for t in np.arange(0.05, 0.96, 0.05):
    m = binary_metrics(yva, p, threshold=t)
    rows.append(dict(threshold=round(t, 2), recall_pneumonia=m['recall_pneumonia'], recall_normal=m['recall_normal'], precision_pneumonia=m['precision_pneumonia'], macro_f1=m['macro_f1'], fn=m['fn'], fp=m['fp']))
th = pd.DataFrame(rows); display(th.round(3))
ok = th[th.recall_pneumonia >= 0.98]
CHOSEN = float(ok.threshold.max()) if len(ok) else float(th.sort_values('recall_pneumonia', ascending=False).threshold.iloc[0])
print('chosen threshold (highest cut-off with val recall_pneumonia >= 0.98):', CHOSEN)
print(th[th.threshold == round(CHOSEN, 2)].round(4).to_dict('records'))
plt.plot(th.threshold, th.recall_pneumonia, label='recall pneumonia'); plt.plot(th.threshold, th.recall_normal, label='recall normal')
plt.axvline(CHOSEN, color='k', ls='--'); plt.legend(); plt.xlabel('threshold'); plt.title('Threshold trade-off (val)'); plt.show()

**How to read the threshold:** lowering it from 0.5 makes the model say "pneumonia" more often. Recall on pneumonia goes up (fewer missed cases) and recall on normal goes down (more healthy children flagged). That is the right trade when a missed case is the costly error, but it must be justified in the report and the threshold is fixed on val before anyone looks at the test set.